In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import warnings

In [2]:
warnings.filterwarnings('ignore')


In [ ]:
"""
Complete Spatial Hedonic Price Model Analysis
Analyzing impact of street tree canopy on property prices with spatial autocorrelation
"""

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Spatial analysis libraries
try:
    import libpysal
    from libpysal.weights import KNN, DistanceBand
    from esda.moran import Moran
    from spreg import OLS, ML_Lag, ML_Error, GM_Lag
    from spreg.ml_error_regimes import ML_Error_Regimes
except ImportError:
    print("Installing required spatial libraries...")
    print("Run: pip install libpysal esda spreg")

# ============================================================================
# STEP 1: LOAD AND PREPARE DATA
# ============================================================================

def load_and_prepare_data(filepath):
    """
    Load your GeoDataFrame with property data
    """
    print("="*80)
    print("STEP 1: LOADING AND PREPARING DATA")
    print("="*80)
    
    # Load the data
    print(f"\n   Loading data from: {filepath}")
    gdf = gpd.read_file(filepath)
    
    if 'log_price' not in gdf.columns:
        gdf['log_price'] = np.log(gdf['GrossSalePrice'])
    
    print(f"   ✓ Loaded {len(gdf):,} properties")
    print(f"   ✓ CRS: {gdf.crs}")
    print(f"   ✓ Columns: {len(gdf.columns)}")
    
    # Check for required columns
    required_cols = ['log_price', 'geometry']
    missing = [col for col in required_cols if col not in gdf.columns]
    if missing:
        print(f"\n   ⚠️  Missing required columns: {missing}")
    else:
        print(f"   ✓ All required columns present")
    
    return gdf

# ============================================================================
# STEP 2: EXPLORATORY DATA ANALYSIS
# ============================================================================

def exploratory_analysis(gdf):
    """
    Comprehensive EDA including distributions, correlations, and spatial patterns
    """
    print("\n" + "="*80)
    print("STEP 2: EXPLORATORY DATA ANALYSIS")
    print("="*80)
    
    # 2.1 Summary statistics
    print("\n2.1 Summary Statistics")
    print("-" * 40)
    
    key_vars = ['log_price', 'GrossSalePrice', 'AgeAtSale', 'LandArea', 
                'TotalFloorArea', 'water_DIST', 'bus_DIST', 'CBD_DIST']
    print(gdf[key_vars].describe())
    
    # 2.2 Distribution of outcome variable
    print("\n2.2 Outcome Variable Distribution")
    print("-" * 40)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Histogram
    axes[0].hist(gdf['log_price'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Log Price')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Log Property Price')
    axes[0].axvline(gdf['log_price'].mean(), color='red', 
                    linestyle='--', label='Mean')
    axes[0].legend()
    
    # Q-Q plot
    stats.probplot(gdf['log_price'], dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q Plot')
    
    # Box plot
    axes[2].boxplot(gdf['log_price'])
    axes[2].set_ylabel('Log Price')
    axes[2].set_title('Box Plot (Outlier Detection)')
    
    plt.tight_layout()
    plt.savefig('01_outcome_distribution.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 01_outcome_distribution.png")
    plt.close()
    
    # 2.3 Correlation analysis
    print("\n2.3 Correlation Analysis")
    print("-" * 40)
    
    # All numeric columns
    numeric_cols = gdf.select_dtypes(include=[np.number]).columns.tolist()
    if 'geometry' in numeric_cols:
        numeric_cols.remove('geometry')
    
    # Correlation matrix
    corr_matrix = gdf[numeric_cols].corr()
    
    # Plot correlation heatmap for key variables
    key_vars_extended = ['log_price', 'AgeAtSale', 'LandArea', 'TotalFloorArea',
                         'water_DIST', 'bus_DIST', 'CBD_DIST', 'Median_Income',
                         'RnkIMDNoEm', 'RnkIMDNoIn', 'RnkIMDNoCr']
    
    # Add canopy variables
    canopy_vars = [col for col in gdf.columns if col.startswith('canopy_')]
    key_vars_extended.extend(canopy_vars)
    
    # Filter to existing columns
    key_vars_extended = [v for v in key_vars_extended if v in corr_matrix.columns]
    
    plt.figure(figsize=(16, 14))
    sns.heatmap(corr_matrix.loc[key_vars_extended, key_vars_extended], 
                annot=False, cmap='coolwarm', center=0, 
                vmin=-1, vmax=1, square=True)
    plt.title('Correlation Matrix: Key Variables', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('02_correlation_matrix.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 02_correlation_matrix.png")
    plt.close()
    
    # Check multicollinearity - IMD variables
    print("\n   Correlation among IMD variables:")
    imd_vars = [col for col in gdf.columns if 'RnkIMDNo' in col]
    if len(imd_vars) > 0:
        imd_corr = gdf[imd_vars].corr()
        print(imd_corr)
        print(f"\n   ⚠️  Average correlation: {imd_corr.values[np.triu_indices_from(imd_corr.values, k=1)].mean():.3f}")
    
    # Check multicollinearity - Canopy variables
    print("\n   Correlation among canopy distance bands:")
    if len(canopy_vars) > 0:
        canopy_corr = gdf[canopy_vars].corr()
        print(canopy_corr)
        print(f"\n   ⚠️  Average correlation: {canopy_corr.values[np.triu_indices_from(canopy_corr.values, k=1)].mean():.3f}")
    
    # 2.4 Spatial distribution
    print("\n2.4 Spatial Distribution")
    print("-" * 40)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Property prices
    gdf.plot(column='log_price', cmap='YlOrRd', legend=True, 
             ax=axes[0], markersize=1, alpha=0.6)
    axes[0].set_title('Spatial Distribution: Log Property Price', fontweight='bold')
    axes[0].axis('off')
    
    # Canopy (example: 100-150m band)
    if 'canopy_100_150' in gdf.columns:
        gdf.plot(column='canopy_100_150', cmap='Greens', legend=True, 
                 ax=axes[1], markersize=1, alpha=0.6)
        axes[1].set_title('Spatial Distribution: Canopy (100-150m)', fontweight='bold')
        axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('03_spatial_distribution.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 03_spatial_distribution.png")
    plt.close()
    
    return key_vars_extended, canopy_vars, imd_vars

# ============================================================================
# STEP 3: VARIABLE SELECTION & FEATURE ENGINEERING
# ============================================================================

def feature_engineering(gdf, imd_vars):
    """
    Handle multicollinearity and create aggregated features
    """
    print("\n" + "="*80)
    print("STEP 3: VARIABLE SELECTION & FEATURE ENGINEERING")
    print("="*80)
    
    gdf_model = gdf.copy()
    
    # 3.1 Handle IMD multicollinearity - create composite index
    print("\n3.1 Creating Composite Deprivation Index")
    print("-" * 40)
    
    if len(imd_vars) > 0:
        # Standardize IMD variables
        scaler = StandardScaler()
        imd_scaled = scaler.fit_transform(gdf_model[imd_vars])
        
        # Create composite (mean of standardized ranks)
        gdf_model['IMD_composite'] = imd_scaled.mean(axis=1)
        print(f"   ✓ Created IMD_composite from {len(imd_vars)} deprivation measures")
    
    # 3.2 Aggregate canopy into distance categories
    print("\n3.2 Creating Aggregated Canopy Variables")
    print("-" * 40)
    
    canopy_cols = [col for col in gdf_model.columns if col.startswith('canopy_')]
    
    if len(canopy_cols) > 0:
        # Keep individual bands up to 200m: 0-25, 25-50, 50-75, 75-100, 100-150, 150-200
        keep_individual = ['canopy_0_25', 'canopy_25_50', 'canopy_50_75', 
                          'canopy_75_100', 'canopy_100_150', 'canopy_150_200']
        
        available_individual = [c for c in keep_individual if c in canopy_cols]
        print(f"   ✓ Keeping individual bands (0-200m): {', '.join(available_individual)}")
        
        # Aggregate 200-400m
        distant = [c for c in canopy_cols if any(x in c for x in ['200_250', '250_300', '300_350', '350_400'])]
        if distant:
            gdf_model['canopy_200_400'] = gdf_model[distant].sum(axis=1)
            print(f"   ✓ Created aggregated band: canopy_200_400 (from {len(distant)} bands)")
    
    else:
        print("   ⚠️  No canopy variables found")
    
    # 3.3 Log-transform distance variables (if appropriate)
    print("\n3.3 Log-Transforming Distance Variables")
    print("-" * 40)
    
    dist_vars = ['water_DIST', 'bus_DIST', 'CBD_DIST', 'cycleways_DIST']
    for var in dist_vars:
        if var in gdf_model.columns:
            # Add small constant to avoid log(0)
            gdf_model[f'log_{var}'] = np.log(gdf_model[var] + 1)
            print(f"   ✓ Created log_{var}")
    
    # 3.4 Interaction terms (example: canopy × income)
    print("\n3.4 Creating Interaction Terms")
    print("-" * 40)
    
    if 'canopy_0_25' in gdf_model.columns and 'Median_Income' in gdf_model.columns:
        gdf_model['canopy_0_25_x_income'] = gdf_model['canopy_0_25'] * gdf_model['Median_Income']
        print("   ✓ Created canopy_0_25 × Median_Income interaction")
    else:
        print("   ⚠️  Cannot create interaction (missing variables)")
    
    return gdf_model

# ============================================================================
# STEP 4: BASELINE OLS MODEL
# ============================================================================

def fit_baseline_ols(gdf):
    """
    Fit baseline OLS hedonic model
    """
    print("\n" + "="*80)
    print("STEP 4: BASELINE OLS HEDONIC MODEL")
    print("="*80)
    
    # Define model specification
    print("\n4.1 Model Specification")
    print("-" * 40)
    
    # Core hedonic variables
    X_vars = [
        'AgeAtSale', 'LandArea', 'TotalFloorArea',
        'log_water_DIST', 'log_bus_DIST', 'log_CBD_DIST',
        'Median_Income', 'IMD_composite',
        'canopy_0_25', 'canopy_25_50', 'canopy_50_75', 'canopy_75_100',
        'canopy_100_150', 'canopy_150_200', 'canopy_200_400',
        'year_2018', 'year_2019'
    ]
    
    # Filter to existing columns
    X_vars = [v for v in X_vars if v in gdf.columns]
    
    print(f"   Dependent variable: log_price")
    print(f"   Independent variables ({len(X_vars)}):")
    for var in X_vars:
        print(f"      • {var}")
    
    # Prepare data
    y = gdf['log_price'].values.reshape(-1, 1)
    X = gdf[X_vars].values
    
    # Add constant
    X_const = np.hstack([np.ones((X.shape[0], 1)), X])
    var_names = ['Constant'] + X_vars
    
    # Fit OLS using spreg
    print("\n4.2 Estimating OLS Model")
    print("-" * 40)
    
    ols_model = OLS(y, X_const, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("OLS REGRESSION RESULTS")
    print("="*60)
    print(ols_model.summary)
    
    # Extract residuals for spatial analysis
    residuals = ols_model.u
    
    return ols_model, residuals, X_vars

# ============================================================================
# STEP 5: SPATIAL WEIGHTS MATRIX
# ============================================================================

def create_spatial_weights(gdf, method='knn', k=8, distance_threshold=500):
    """
    Create spatial weights matrix
    """
    print("\n" + "="*80)
    print("STEP 5: CONSTRUCTING SPATIAL WEIGHTS MATRIX")
    print("="*80)
    
    # Get coordinates
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
    
    if method == 'knn':
        print(f"\n5.1 K-Nearest Neighbors (k={k})")
        print("-" * 40)
        w = KNN.from_array(coords, k=k)
        print(f"   ✓ Created KNN weights matrix")
        cardinalities = list(w.cardinalities.values())
        print(f"   • Average neighbors: {np.mean(cardinalities):.2f}")
        
    elif method == 'distance':
        print(f"\n5.1 Distance Band (threshold={distance_threshold}m)")
        print("-" * 40)
        w = DistanceBand.from_array(coords, threshold=distance_threshold)
        print(f"   ✓ Created distance-based weights matrix")
        cardinalities = list(w.cardinalities.values())
        print(f"   • Average neighbors: {np.mean(cardinalities):.2f}")
        print(f"   • Min neighbors: {min(cardinalities)}")
        print(f"   • Max neighbors: {max(cardinalities)}")
    
    # Row-standardize
    w.transform = 'r'
    print(f"   ✓ Row-standardized weights")
    
    return w

# ============================================================================
# STEP 6: TEST FOR SPATIAL AUTOCORRELATION
# ============================================================================

def test_spatial_autocorrelation(residuals, w, gdf):
    """
    Test for spatial autocorrelation in OLS residuals
    """
    print("\n" + "="*80)
    print("STEP 6: TESTING FOR SPATIAL AUTOCORRELATION")
    print("="*80)
    
    # Moran's I test
    print("\n6.1 Global Moran's I Test")
    print("-" * 40)
    
    moran = Moran(residuals.flatten(), w)
    
    print(f"   Moran's I statistic: {moran.I:.4f}")
    print(f"   Expected I: {moran.EI:.4f}")
    print(f"   Variance: {moran.VI_norm:.6f}")
    print(f"   Z-score: {moran.z_norm:.4f}")
    print(f"   P-value: {moran.p_norm:.6f}")
    
    if moran.p_norm < 0.01:
        print(f"\n   *** STRONG spatial autocorrelation detected (p < 0.01)")
        print(f"   *** Spatial model is necessary!")
    elif moran.p_norm < 0.05:
        print(f"\n   ** Significant spatial autocorrelation (p < 0.05)")
    else:
        print(f"\n   No significant spatial autocorrelation detected")
    
    # 6.2 Map residuals
    print("\n6.2 Mapping OLS Residuals")
    print("-" * 40)
    
    gdf_plot = gdf.copy()
    gdf_plot['ols_residuals'] = residuals
    
    fig, ax = plt.subplots(figsize=(12, 10))
    gdf_plot.plot(column='ols_residuals', cmap='RdBu', legend=True,
                  ax=ax, markersize=2, alpha=0.6,
                  vmin=-residuals.std()*2, vmax=residuals.std()*2)
    ax.set_title('OLS Residuals (Spatial Pattern)', fontweight='bold', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('04_ols_residuals_map.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 04_ols_residuals_map.png")
    plt.close()
    
    # 6.3 Moran scatterplot
    print("\n6.3 Moran Scatterplot")
    print("-" * 40)
    
    # Create Moran scatterplot manually
    from libpysal.weights.spatial_lag import lag_spatial
    
    # Standardize residuals
    residuals_std = (residuals - residuals.mean()) / residuals.std()
    
    # Calculate spatial lag of standardized residuals
    residuals_lag = lag_spatial(w, residuals_std.flatten())
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Scatter plot
    ax.scatter(residuals_std, residuals_lag, alpha=0.5, s=20)
    
    # Add regression line
    from scipy.stats import linregress
    slope, intercept, r_value, p_value, std_err = linregress(residuals_std.flatten(), residuals_lag)
    line_x = np.array([residuals_std.min(), residuals_std.max()])
    line_y = slope * line_x + intercept
    ax.plot(line_x, line_y, 'r-', linewidth=2, label=f'Slope = {slope:.3f}')
    
    # Add reference lines
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
    
    # Labels and title
    ax.set_xlabel('Standardized Residuals', fontsize=12)
    ax.set_ylabel('Spatial Lag of Standardized Residuals', fontsize=12)
    ax.set_title(f"Moran's I Scatterplot\nI = {moran.I:.4f}, p = {moran.p_norm:.4f}", 
                 fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('05_moran_scatterplot.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 05_moran_scatterplot.png")
    plt.close()
    
    return moran

# ============================================================================
# STEP 7: FIT SPATIAL MODELS
# ============================================================================

def fit_spatial_models(y, X_const, var_names, w):
    """
    Fit spatial lag and spatial error models
    """
    print("\n" + "="*80)
    print("STEP 7: ESTIMATING SPATIAL MODELS")
    print("="*80)
    
    # 7.1 Spatial Lag Model (SAR)
    print("\n7.1 Spatial Lag Model (SAR)")
    print("-" * 40)
    print("   Estimating via Maximum Likelihood...")
    
    lag_model = ML_Lag(y, X_const, w=w, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("SPATIAL LAG MODEL RESULTS")
    print("="*60)
    print(lag_model.summary)
    
    # 7.2 Spatial Error Model (SEM)
    print("\n7.2 Spatial Error Model (SEM)")
    print("-" * 40)
    print("   Estimating via Maximum Likelihood...")
    
    error_model = ML_Error(y, X_const, w=w, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("SPATIAL ERROR MODEL RESULTS")
    print("="*60)
    print(error_model.summary)
    
    return lag_model, error_model

# ============================================================================
# STEP 8: MODEL COMPARISON
# ============================================================================

def compare_models(ols_model, lag_model, error_model):
    """
    Compare OLS vs spatial models
    """
    print("\n" + "="*80)
    print("STEP 8: MODEL COMPARISON")
    print("="*80)
    
    # Extract fit statistics
    models = {
        'OLS': {
            'R²': ols_model.r2,
            'Adj. R²': ols_model.ar2,
            'AIC': ols_model.aic,
            'Log-Likelihood': ols_model.logll
        },
        'Spatial Lag': {
            'Pseudo R²': lag_model.pr2,
            'AIC': lag_model.aic,
            'Log-Likelihood': lag_model.logll,
            'Rho (ρ)': lag_model.rho
        },
        'Spatial Error': {
            'Pseudo R²': error_model.pr2,
            'AIC': error_model.aic,
            'Log-Likelihood': error_model.logll,
            'Lambda (λ)': error_model.lam
        }
    }
    
    # Create comparison table
    comparison_df = pd.DataFrame(models).T
    
    print("\n8.1 Model Fit Comparison")
    print("-" * 40)
    print(comparison_df.to_string())
    
    # Best model by AIC
    best_model_name = comparison_df['AIC'].idxmin()
    print(f"\n   *** Best model by AIC: {best_model_name}")
    
    # Likelihood ratio tests
    print("\n8.2 Likelihood Ratio Tests")
    print("-" * 40)
    
    # LR test: OLS vs Lag
    lr_lag = 2 * (lag_model.logll - ols_model.logll)
    print(f"   OLS vs Spatial Lag:")
    print(f"      LR statistic: {lr_lag:.4f}")
    print(f"      (Chi-square with 1 df, critical value at 0.05 = 3.841)")
    if lr_lag > 3.841:
        print(f"      *** Spatial Lag significantly better than OLS")
    
    # LR test: OLS vs Error
    lr_error = 2 * (error_model.logll - ols_model.logll)
    print(f"\n   OLS vs Spatial Error:")
    print(f"      LR statistic: {lr_error:.4f}")
    if lr_error > 3.841:
        print(f"      *** Spatial Error significantly better than OLS")
    
    return comparison_df, best_model_name

# ============================================================================
# STEP 9: INTERPRET CANOPY EFFECTS
# ============================================================================

def interpret_canopy_effects(lag_model, error_model, X_vars):
    """
    Interpret coefficients for canopy variables
    """
    print("\n" + "="*80)
    print("STEP 9: INTERPRETING CANOPY EFFECTS")
    print("="*80)
    
    # Get canopy variable indices and names
    canopy_indices = [i for i, var in enumerate(['Constant'] + X_vars) 
                      if 'canopy' in var.lower()]
    canopy_vars = [(['Constant'] + X_vars)[i] for i in canopy_indices]
    
    if len(canopy_vars) == 0:
        print("   No canopy variables found in model")
        return
    
    print("\n9.1 Canopy Coefficients (Spatial Lag Model)")
    print("-" * 40)
    
    for idx, var in zip(canopy_indices, canopy_vars):
        coef = lag_model.betas[idx][0]
        # For log-linear model: percentage change = (exp(coef) - 1) * 100
        pct_change = (np.exp(coef) - 1) * 100
        
        print(f"\n   {var}:")
        print(f"      Coefficient: {coef:.6f}")
        print(f"      Percentage effect: {pct_change:.2f}%")
        print(f"      Interpretation: A 1-unit increase in {var} is associated")
        print(f"                      with a {pct_change:.2f}% change in property price")
    
    # For Spatial Lag: compute direct and indirect effects
    if hasattr(lag_model, 'rho'):
        print("\n9.2 Direct vs. Indirect Effects (Spillovers)")
        print("-" * 40)
        print(f"   Spatial parameter (ρ) = {lag_model.rho:.4f}")
        print("\n   Note: In Spatial Lag models, effects are decomposed into:")
        print("      • Direct effect: impact on own property")
        print("      • Indirect effect: spillover from neighboring properties")
        print("      • Total effect: direct + indirect")
        print("\n   For precise calculation, use: impacts = lag_model.impacts()")
        print("   (requires additional computation)")
    
    print("\n9.3 Canopy Coefficients (Spatial Error Model)")
    print("-" * 40)
    
    for idx, var in zip(canopy_indices, canopy_vars):
        coef = error_model.betas[idx][0]
        pct_change = (np.exp(coef) - 1) * 100
        
        print(f"\n   {var}:")
        print(f"      Coefficient: {coef:.6f}")
        print(f"      Percentage effect: {pct_change:.2f}%")

# ============================================================================
# STEP 10: RESIDUAL DIAGNOSTICS
# ============================================================================

def residual_diagnostics(lag_model, w, gdf):
    """
    Check residuals from spatial model for remaining autocorrelation
    """
    print("\n" + "="*80)
    print("STEP 10: RESIDUAL DIAGNOSTICS")
    print("="*80)
    
    # Get residuals from spatial lag model
    residuals = lag_model.u
    
    # Test for remaining spatial autocorrelation
    print("\n10.1 Moran's I Test on Spatial Model Residuals")
    print("-" * 40)
    
    moran_residuals = Moran(residuals.flatten(), w)
    
    print(f"   Moran's I statistic: {moran_residuals.I:.4f}")
    print(f"   Z-score: {moran_residuals.z_norm:.4f}")
    print(f"   P-value: {moran_residuals.p_norm:.6f}")
    
    if moran_residuals.p_norm > 0.05:
        print(f"\n   ✓ No significant spatial autocorrelation in residuals")
        print(f"     Spatial model adequately captures spatial dependence!")
    else:
        print(f"\n   ⚠️  Some spatial autocorrelation remains")
        print(f"     Consider: Spatial Durbin Model or additional spatial predictors")
    
    # Map residuals
    print("\n10.2 Mapping Spatial Model Residuals")
    print("-" * 40)
    
    gdf_plot = gdf.copy()
    gdf_plot['spatial_residuals'] = residuals
    
    fig, ax = plt.subplots(figsize=(12, 10))
    gdf_plot.plot(column='spatial_residuals', cmap='RdBu', legend=True,
                  ax=ax, markersize=2, alpha=0.6,
                  vmin=-residuals.std()*2, vmax=residuals.std()*2)
    ax.set_title('Spatial Lag Model Residuals', fontweight='bold', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('06_spatial_residuals_map.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 06_spatial_residuals_map.png")
    plt.close()
    
    # Histogram of residuals
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Residuals')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Spatial Model Residuals')
    axes[0].axvline(0, color='red', linestyle='--')
    
    stats.probplot(residuals.flatten(), dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q Plot')
    
    plt.tight_layout()
    plt.savefig('07_spatial_residuals_dist.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 07_spatial_residuals_dist.png")
    plt.close()

# ============================================================================
# STEP 11: ROBUSTNESS CHECKS
# ============================================================================

def robustness_checks(y, X_const, var_names, gdf):
    """
    Test sensitivity to different spatial weights specifications
    """
    print("\n" + "="*80)
    print("STEP 11: ROBUSTNESS CHECKS")
    print("="*80)
    
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
    
    print("\n11.1 Alternative Spatial Weights Matrices")
    print("-" * 40)
    
    # Test different k values
    k_values = [5, 8, 10, 15]
    results = []
    
    for k in k_values:
        print(f"\n   Testing k={k} nearest neighbors...")
        w_k = KNN.from_array(coords, k=k)
        w_k.transform = 'r'
        
        lag_k = ML_Lag(y, X_const, w=w_k, name_y='log_price', name_x=var_names)
        
        results.append({
            'k': k,
            'AIC': lag_k.aic,
            'Pseudo R²': lag_k.pr2,
            'Rho': lag_k.rho
        })
    
    results_df = pd.DataFrame(results)
    print("\n   Results across different k values:")
    print(results_df.to_string(index=False))
    
    print("\n11.2 Distance-Based Weights (Alternative)")
    print("-" * 40)
    
    # Test distance thresholds
    distances = [300, 500, 800]
    for dist in distances:
        print(f"\n   Testing distance threshold={dist}m...")
        w_dist = DistanceBand.from_array(coords, threshold=dist)
        if min(w_dist.cardinalities.values()) == 0:
            print(f"      ⚠️  Some observations have no neighbors at {dist}m - skipping")
            continue
        w_dist.transform = 'r'
        
        lag_dist = ML_Lag(y, X_const, w=w_dist, name_y='log_price', name_x=var_names)
        print(f"      AIC: {lag_dist.aic:.2f}")
        print(f"      Avg neighbors: {np.mean(list(w_dist.cardinalities.values())):.1f}")

# ============================================================================
# STEP 12: EXPORT RESULTS
# ============================================================================

def export_results(ols_model, lag_model, error_model, X_vars, gdf):
    """
    Export regression tables and predictions
    """
    print("\n" + "="*80)
    print("STEP 12: EXPORTING RESULTS")
    print("="*80)
    
    # 12.1 Create regression table
    print("\n12.1 Creating Regression Table")
    print("-" * 40)
    
    var_names = ['Constant'] + X_vars
    n_vars = len(var_names)
    
    # Extract coefficients and standard errors (only for X variables, not spatial params)
    results_dict = {
        'Variable': var_names,
        'OLS_Coef': ols_model.betas.flatten()[:n_vars],
        'OLS_SE': np.sqrt(ols_model.vm.diagonal()[:n_vars]),
        'Lag_Coef': lag_model.betas.flatten()[:n_vars],
        'Lag_SE': np.sqrt(lag_model.vm.diagonal()[:n_vars]),
        'Error_Coef': error_model.betas.flatten()[:n_vars],
        'Error_SE': np.sqrt(error_model.vm.diagonal()[:n_vars])
    }
    
    results_table = pd.DataFrame(results_dict)
    
    # Add significance stars
    for model in ['OLS', 'Lag', 'Error']:
        results_table[f'{model}_tstat'] = (results_table[f'{model}_Coef'] / 
                                           results_table[f'{model}_SE'])
        results_table[f'{model}_sig'] = results_table[f'{model}_tstat'].apply(
            lambda t: '***' if abs(t) > 2.576 else ('**' if abs(t) > 1.96 else 
                     ('*' if abs(t) > 1.645 else ''))
        )
    
    # Add spatial parameters separately
    spatial_params = pd.DataFrame({
        'Variable': ['Rho (ρ)', 'Lambda (λ)'],
        'OLS_Coef': [np.nan, np.nan],
        'OLS_SE': [np.nan, np.nan],
        'Lag_Coef': [lag_model.rho, np.nan],
        'Lag_SE': [np.sqrt(lag_model.vm[-1, -1]), np.nan],
        'Error_Coef': [np.nan, error_model.lam],
        'Error_SE': [np.nan, np.sqrt(error_model.vm[-1, -1])],
        'OLS_tstat': [np.nan, np.nan],
        'OLS_sig': ['', ''],
        'Lag_tstat': [lag_model.rho / np.sqrt(lag_model.vm[-1, -1]), np.nan],
        'Lag_sig': ['***' if abs(lag_model.rho / np.sqrt(lag_model.vm[-1, -1])) > 2.576 else '**' if abs(lag_model.rho / np.sqrt(lag_model.vm[-1, -1])) > 1.96 else '*' if abs(lag_model.rho / np.sqrt(lag_model.vm[-1, -1])) > 1.645 else '', ''],
        'Error_tstat': [np.nan, error_model.lam / np.sqrt(error_model.vm[-1, -1])],
        'Error_sig': ['', '***' if abs(error_model.lam / np.sqrt(error_model.vm[-1, -1])) > 2.576 else '**' if abs(error_model.lam / np.sqrt(error_model.vm[-1, -1])) > 1.96 else '*' if abs(error_model.lam / np.sqrt(error_model.vm[-1, -1])) > 1.645 else '']
    })
    
    # Combine
    results_table = pd.concat([results_table, spatial_params], ignore_index=True)
    
    # Export to CSV
    results_table.to_csv('regression_results_table.csv', index=False)
    print("   ✓ Saved: regression_results_table.csv")
    
    # Print formatted table
    print("\n   Key Variables Summary:")
    print("-" * 40)
    display_cols = ['Variable', 'OLS_Coef', 'OLS_sig', 'Lag_Coef', 'Lag_sig', 
                    'Error_Coef', 'Error_sig']
    canopy_rows = results_table[results_table['Variable'].str.contains('canopy|Rho|Lambda', na=False)]
    print(canopy_rows[display_cols].to_string(index=False))
    
    # 12.2 Export predictions
    print("\n12.2 Exporting Predictions")
    print("-" * 40)
    
    gdf_export = gdf.copy()
    gdf_export['predicted_log_price'] = lag_model.predy.flatten()
    gdf_export['predicted_price'] = np.exp(lag_model.predy.flatten())
    gdf_export['residual'] = lag_model.u.flatten()
    
    # Export to shapefile or GeoPackage
    try:
        gdf_export.to_file('property_predictions.gpkg', driver='GPKG')
        print("   ✓ Saved: property_predictions.gpkg")
    except:
        gdf_export.to_file('property_predictions.shp')
        print("   ✓ Saved: property_predictions.shp")
    
    # Also save as CSV (without geometry)
    export_cols = ['log_price', 'GrossSalePrice', 'predicted_log_price', 
                   'predicted_price', 'residual'] + X_vars
    export_cols = [c for c in export_cols if c in gdf_export.columns]
    gdf_export[export_cols].to_csv('property_predictions.csv', index=False)
    print("   ✓ Saved: property_predictions.csv")
    
    # 12.3 Summary statistics
    print("\n12.3 Prediction Quality")
    print("-" * 40)
    
    mae = np.mean(np.abs(gdf_export['residual']))
    rmse = np.sqrt(np.mean(gdf_export['residual']**2))
    
    print(f"   Mean Absolute Error (MAE): {mae:.4f}")
    print(f"   Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"   Pseudo R²: {lag_model.pr2:.4f}")
    
    # Actual vs Predicted plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Scatter plot
    axes[0].scatter(gdf_export['log_price'], gdf_export['predicted_log_price'], 
                    alpha=0.3, s=10)
    axes[0].plot([gdf_export['log_price'].min(), gdf_export['log_price'].max()],
                 [gdf_export['log_price'].min(), gdf_export['log_price'].max()],
                 'r--', linewidth=2, label='Perfect prediction')
    axes[0].set_xlabel('Actual Log Price', fontsize=11)
    axes[0].set_ylabel('Predicted Log Price', fontsize=11)
    axes[0].set_title('Actual vs Predicted Prices', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Residual plot
    axes[1].scatter(gdf_export['predicted_log_price'], gdf_export['residual'], 
                    alpha=0.3, s=10)
    axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Predicted Log Price', fontsize=11)
    axes[1].set_ylabel('Residuals', fontsize=11)
    axes[1].set_title('Residual Plot', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('08_prediction_diagnostics.png', dpi=300, bbox_inches='tight')
    print("   ✓ Saved: 08_prediction_diagnostics.png")
    plt.close()

# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================

def main(filepath):
    """
    Run complete spatial hedonic analysis pipeline
    """
    print("\n")
    print("="*80)
    print(" SPATIAL HEDONIC PRICE ANALYSIS: STREET TREE CANOPY EFFECTS")
    print("="*80)
    print("\n")
    
    # Step 1: Load data
    gdf = load_and_prepare_data(filepath)
    
    # Step 2: EDA
    key_vars, canopy_vars, imd_vars = exploratory_analysis(gdf)
    
    # Step 3: Feature engineering
    gdf_model = feature_engineering(gdf, imd_vars)
    
    # Step 4: Baseline OLS
    ols_model, ols_residuals, X_vars = fit_baseline_ols(gdf_model)
    
    # Step 5: Spatial weights
    w = create_spatial_weights(gdf_model, method='knn', k=8)
    
    # Step 6: Test spatial autocorrelation
    moran = test_spatial_autocorrelation(ols_residuals, w, gdf_model)
    
    # Prepare data for spatial models
    y = gdf_model['log_price'].values.reshape(-1, 1)
    X = gdf_model[X_vars].values
    X_const = np.hstack([np.ones((X.shape[0], 1)), X])
    var_names = ['Constant'] + X_vars
    
    # Step 7: Fit spatial models
    lag_model, error_model = fit_spatial_models(y, X_const, var_names, w)
    
    # Step 8: Model comparison
    comparison_df, best_model = compare_models(ols_model, lag_model, error_model)
    
    # Step 9: Interpret canopy effects
    interpret_canopy_effects(lag_model, error_model, X_vars)
    
    # Step 10: Residual diagnostics
    residual_diagnostics(lag_model, w, gdf_model)
    
    # Step 11: Robustness checks
    robustness_checks(y, X_const, var_names, gdf_model)
    
    # Step 12: Export results
    export_results(ols_model, lag_model, error_model, X_vars, gdf_model)
    
    print("\n")
    print("="*80)
    print(" ANALYSIS COMPLETE!")
    print("="*80)
    print("\nGenerated outputs:")
    print("   Figures:")
    print("      • 01_outcome_distribution.png")
    print("      • 02_correlation_matrix.png")
    print("      • 03_spatial_distribution.png")
    print("      • 04_ols_residuals_map.png")
    print("      • 05_moran_scatterplot.png")
    print("      • 06_spatial_residuals_map.png")
    print("      • 07_spatial_residuals_dist.png")
    print("\n   Tables:")
    print("      • regression_results_table.csv")
    print("      • property_predictions.csv")
    print("      • property_predictions.shp")
    print("\n")
    print("="*80)
    print(f" BEST MODEL: {best_model}")
    print("="*80)
    print("\n")

# ============================================================================
# RUN THE ANALYSIS
# ============================================================================

if __name__ == "__main__":
    # Your actual file path
    filepath = "../output/property_all.gpkg"
    
    # Run the complete pipeline
    main(filepath)
    
    print("\n" + "="*80)
    print(" NEXT STEPS & RECOMMENDATIONS")
    print("="*80)
    print("""
    1. ADVANCED MODELS (if needed):
       • Spatial Durbin Model (SDM) - captures spillover effects
       • Geographically Weighted Regression (GWR) - spatially varying coefficients
       • Use mgwr library for GWR
    
    2. POLICY IMPLICATIONS:
       • Calculate willingness-to-pay for canopy
       • Identify optimal distance bands for tree planting
       • Quantify equity impacts across income groups
    
    3. SENSITIVITY ANALYSIS:
       • Test alternative canopy aggregations
       • Include interaction terms (canopy × deprivation)
       • Subset analysis by property type or neighborhood
    
    4. ENDOGENEITY CONCERNS:
       • Consider instrumental variables if reverse causality suspected
       • Use lagged canopy data if available
       • Panel data models if multiple time periods available
    
    5. VALIDATION:
       • Out-of-sample prediction (hold-out test set)
       • Cross-validation
       • Compare predictions to actual recent sales
    """)
    print("="*80)



 SPATIAL HEDONIC PRICE ANALYSIS: STREET TREE CANOPY EFFECTS


STEP 1: LOADING AND PREPARING DATA

   Loading data from: ../output/hedonic_gdf.gpkg
   ✓ Loaded 12,431 properties
   ✓ CRS: EPSG:2193
   ✓ Columns: 29
   ✓ All required columns present

STEP 2: EXPLORATORY DATA ANALYSIS

2.1 Summary Statistics
----------------------------------------
          log_price  GrossSalePrice     AgeAtSale       LandArea  \
count  12431.000000    1.243100e+04  12431.000000   12431.000000   
mean      13.150518    5.469447e+05     49.228542     769.294667   
std        0.337569    2.159642e+05     30.894695    2626.946951   
min       11.608236    1.100000e+05      2.000000      53.000000   
25%       12.926348    4.110000e+05     22.000000     502.000000   
50%       13.108264    4.930000e+05     52.000000     627.000000   
75%       13.342302    6.230000e+05     72.000000     751.000000   
max       14.357835    1.720000e+06    144.000000  125230.000000   

       TotalFloorArea    water_DIST  